# H₂を持つクラスタの個数：Colab解析

目的：H₂を1個以上含む／含まないクラスタを数える。対象はL=128、目標φ=0.43の100試料。最大クラスタを含む結果と、1個除いた結果を出力します。

H₂は「同じ2分子がちょうど2本の結合を持つペア」。H₂なしでも、他の閉路がないとは限りません。

## 入力の準備（PCで1回だけ）
閉路解析を実行したPCのリポジトリ直下で、以下を実行してください。生成・DFSはやり直しません。キャッシュはDriveの以前の共有ZIPには入っていないため、この受け渡しが必要です。

```bash
python3 -m zipfile -c h2_presence_input.zip data/validation/h2_relation_fine_compact/cycle46_cache/d16eede57d87d37a/*.json
```

作成された **h2_presence_input.zip** を下のセルで選択します。既存のcycle46_share.zipやsamples.csvでは必要な個数を復元できません。

上から順番に実行してください。ファイル選択で待機するのは正常です。選択完了までは進みません。


In [ ]:
from pathlib import Path
import hashlib
import urllib.request
ROOT = Path('/content/h2_presence')
ROOT.mkdir(parents=True, exist_ok=True)
COMMIT = '10fc85cd2576b1f171c6ece564e6b48f3eb90482'
EXPECTED_SHA256 = '446c659f7aca39a5039213684a6326b4f76e81d55dcb087eb8c2ad9acc376ef0'
url = f'https://raw.githubusercontent.com/takakei510/tetra-peg-network/{COMMIT}/scripts/analysis/plot_h2_presence.py'
script_bytes = urllib.request.urlopen(url, timeout=60).read()
assert hashlib.sha256(script_bytes).hexdigest() == EXPECTED_SHA256, '解析コードのハッシュ不一致'
SCRIPT = ROOT / 'plot_h2_presence.py'
SCRIPT.write_bytes(script_bytes)
print('解析コード固定:', COMMIT)


## キャッシュZIPを指定
UPLOADはPCからアップロード、DRIVEはDrive上のZIPを指定する方式です。Drive方式ではINPUT_ZIPに実際の保存先を記入してください。


In [ ]:
MODE = 'UPLOAD'  # 'UPLOAD' or 'DRIVE'
INPUT_ZIP = ''  # DRIVEの場合: /content/drive/MyDrive/.../h2_presence_input.zip

from google.colab import files
if MODE == 'UPLOAD':
    print('h2_presence_input.zip を1ファイル選択してください。選択待ちの間は実行中になります。')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('ZIPを1ファイル選択してください。')
    name, payload = next(iter(uploaded.items()))
    if not name.lower().endswith('.zip'):
        raise ValueError('キャッシュJSONをまとめたZIPを選択してください。')
    zip_path = ROOT / 'uploaded_cache.zip'
    zip_path.write_bytes(payload)
    del uploaded, payload
elif MODE == 'DRIVE':
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = Path(INPUT_ZIP)
    if not zip_path.is_file():
        raise FileNotFoundError('INPUT_ZIPにDrive内のZIPのパスを指定してください。')
else:
    raise ValueError('MODEはUPLOADまたはDRIVEです。')
print('入力ZIP:', zip_path)
print('入力SHA256:', hashlib.sha256(zip_path.read_bytes()).hexdigest())


In [ ]:
import json
import math
import tempfile
from zipfile import ZipFile

def stage_cache(source_zip, parent):
    # ZIPのパスで直接展開せず、検証した試料を新規フォルダに書き出す。
    cache = Path(tempfile.mkdtemp(prefix='cache_', dir=parent))
    seen = set()
    with ZipFile(source_zip) as z:
        for info in z.infolist():
            if info.is_dir() or not info.filename.endswith('.json'):
                continue
            if info.file_size > 100_000_000:
                raise ValueError('キャッシュJSONが想定より大きいです。')
            rec = json.loads(z.read(info))
            sample = rec.get('sample', {})
            if int(sample.get('L', -1)) != 128 or not math.isclose(float(sample.get('target_phi', -1)), .43, abs_tol=1e-9):
                continue
            case = sample.get('case_id', '')
            if not case or Path(case).name != case or case in seen:
                raise ValueError('試料IDが不正または重複しています。')
            if rec.get('version') != 'cycle46-v1' or not isinstance(rec.get('clusters'), list):
                raise ValueError('必要なクラスタ別キャッシュではありません。')
            seen.add(case)
            (cache / (case + '.json')).write_text(json.dumps(rec), encoding='utf-8')
    if len(seen) != 100:
        raise ValueError(f'100試料のキャッシュが必要ですが、{len(seen)}試料でした。cycle46_share.zipは入力に使えません。')
    return cache

CACHE = stage_cache(zip_path, ROOT)
print('入力キャッシュ:', CACHE, '100試料。次のセルで個数・試料の整合性も確認します。')


In [ ]:
import subprocess
import sys
cmd = [sys.executable, str(SCRIPT), '--cache', str(CACHE), '--output', str(ROOT / 'output')]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode:
    print(proc.stderr)
    raise RuntimeError('集計に失敗しました。上記のエラーを確認してください。')
OUTPUT = Path(next(line.removeprefix('OUTPUT: ').strip() for line in proc.stdout.splitlines() if line.startswith('OUTPUT: ')))
from IPython.display import display, Image
for name in ['01_H2_presence_totals.png', '02_H2_presence_by_size.png']:
    display(Image(filename=str(OUTPUT / name)))


## 図の読み方と集計方法
- 図1：全クラスタ／最大クラスタ1個除外について、H₂なし・ありの1試料平均個数。
- 図2上：サイズ別の平均個数。区間が存在しない試料を0個として含め、100試料で平均。
- 図2下：試料ごとに「H₂ありクラスタ数÷対象区間の全クラスタ数」を計算し、区間がある試料間で等重み平均。平均±SE。数字は寄与試料数。50試料以上の区間だけを表示します。50は表示上の目安で精度の保証ではありません。
- 孤立分子は全体個数とCSVに含めます。サイズ別図は2〜512分子を表示し、全サイズの結果はCSVに保存します。
- 既存6ページの色は末端割合の平均でした。本図の「H₂ありクラスタ割合」とは別の測定量です。
- 最大サイズが同率ならcluster_id最小の1個を除きます。試料ごとに計数してから平均し、試料を合算した割合とは区別します。
- キャッシュ内のN・N1・K・結合数・H₂数、100試料・seedの重複を検査します。元の結合CSVとの再照合は行いません。元キャッシュの生成時に実施した検証を前提とします。

## 結果を保存・共有
ZIPにはPNG・SVG、試料別CSV、集計CSV、入力ハッシュと定義を含めます。Colabのランタイム終了で消える前にダウンロードしてください。


In [ ]:
files.download(str(OUTPUT / 'h2_presence_share.zip'))
